In [1]:
import os
import numpy as np
import pandas as pd

from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))

dim_company = pd.read_sql("SELECT * FROM dim_company", engine)
pl = pd.read_sql("SELECT * FROM fact_profit_loss", engine)
bs = pd.read_sql("SELECT * FROM fact_balance_sheet", engine)
cf = pd.read_sql("SELECT * FROM fact_cash_flow", engine)
fa = pd.read_sql("SELECT * FROM fact_analysis", engine)

In [3]:
import pandas as pd
import numpy as np

# -------------------------
# Convert numeric columns
# -------------------------

# PL
for col in [
    "sales",
    "net_profit",
    "opm_pct",
    "dividend_payout_pct",
    "eps"
]:
    if col in pl.columns:
        pl[col] = pd.to_numeric(pl[col], errors="coerce")

# BS
for col in [
    "debt_to_equity"
]:
    if col in bs.columns:
        bs[col] = pd.to_numeric(bs[col], errors="coerce")

# CF
for col in [
    "cash_conversion_ratio"
]:
    if col in cf.columns:
        cf[col] = pd.to_numeric(cf[col], errors="coerce")

# FA
for col in [
    "compounded_sales_growth_pct"
]:
    if col in fa.columns:
        fa[col] = pd.to_numeric(fa[col], errors="coerce")

# Company
if "roe_pct" in dim_company.columns:
    dim_company["roe_pct"] = pd.to_numeric(
        dim_company["roe_pct"],
        errors="coerce"
    )

# -------------------------
# Remove TTM rows
# -------------------------

pl_y = pl[
    ~pl["year_label"].astype(str)
    .str.contains("TTM", case=False, na=False)
].copy()

bs_y = bs[
    ~bs["year_label"].astype(str)
    .str.contains("TTM", case=False, na=False)
].copy()

cf_y = cf[
    ~cf["year_label"].astype(str)
    .str.contains("TTM", case=False, na=False)
].copy()

# -------------------------
# Profitability
# -------------------------

profit_feats = (
    pl_y.groupby("symbol")
    .agg(
        avg_opm=("opm_pct", "mean"),
        avg_dividend=("dividend_payout_pct", "mean")
    )
    .reset_index()
)

# -------------------------
# Growth
# -------------------------

growth_feats = (
    fa.loc[
        fa["period_label"] == "3Y",
        ["symbol", "compounded_sales_growth_pct"]
    ]
    .rename(
        columns={
            "compounded_sales_growth_pct":
            "avg_growth_3y"
        }
    )
)

# -------------------------
# Debt
# -------------------------

de_feats = (
    bs_y.groupby("symbol")
    .agg(
        avg_debt_to_equity=("debt_to_equity", "mean")
    )
    .reset_index()
)

# -------------------------
# Cash Flow
# -------------------------

if "cash_conversion_ratio" in cf_y.columns:

    cf_feats = (
        cf_y.groupby("symbol")
        .agg(
            avg_cash_conversion=(
                "cash_conversion_ratio",
                "mean"
            )
        )
        .reset_index()
    )

else:

    cf_feats = pd.DataFrame({
        "symbol": cf_y["symbol"].unique(),
        "avg_cash_conversion": np.nan
    })

# -------------------------
# ROE
# -------------------------

roe_feats = (
    dim_company[
        ["symbol", "roe_pct"]
    ]
    .rename(
        columns={"roe_pct": "avg_roe"}
    )
)

# -------------------------
# Final feature table
# -------------------------

features = (
    dim_company[
        ["symbol", "company_name", "sector"]
    ]
    .merge(profit_feats, on="symbol", how="left")
    .merge(growth_feats, on="symbol", how="left")
    .merge(de_feats, on="symbol", how="left")
    .merge(cf_feats, on="symbol", how="left")
    .merge(roe_feats, on="symbol", how="left")
)

print(features.head())
print(features.shape)

       symbol                             company_name          sector  \
0         ABB                         Abbott India Ltd   Capital Goods   
1  ADANIENSOL               Adani Energy Solutions Ltd          Energy   
2    ADANIENT                    Adani Enterprises Ltd  Infrastructure   
3  ADANIGREEN                   Adani Green Energy Ltd          Energy   
4  ADANIPORTS  Adani Ports & Special Economic Zone Ltd  Infrastructure   

   avg_opm  avg_dividend  avg_growth_3y  avg_debt_to_equity  \
0   17.250     49.416667            NaN            0.019970   
1   52.300      0.000000            NaN            3.004722   
2    8.750      7.666667            NaN            1.777329   
3   68.875      0.000000            NaN            8.294498   
4   60.500     13.000000            NaN            1.276000   

   avg_cash_conversion  avg_roe  
0                  NaN    34.90  
1                  NaN     8.59  
2                  NaN    13.64  
3                  NaN    14.70  
4     

In [4]:
feat_cols = ["avg_opm","avg_roe","avg_growth_3y","avg_debt_to_equity","avg_cash_conversion","avg_dividend"]
features[feat_cols] = features[feat_cols].fillna(features[feat_cols].median())

In [6]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

# Select features
X_df = features[feat_cols].copy()

# Convert all columns to numeric
for col in X_df.columns:
    X_df[col] = pd.to_numeric(X_df[col], errors="coerce")

# Replace inf values
X_df = X_df.replace([np.inf, -np.inf], np.nan)

# Drop columns that are entirely null
X_df = X_df.dropna(axis=1, how="all")

# Fill remaining NaNs with column median
for col in X_df.columns:
    X_df[col] = X_df[col].fillna(X_df[col].median())

# Final check
print("Total NaNs:", X_df.isna().sum().sum())

# Scale
scaler = StandardScaler()
X = scaler.fit_transform(X_df)

# Similarity Matrix
sim = cosine_similarity(X)

sim_df = pd.DataFrame(
    sim,
    index=features["symbol"],
    columns=features["symbol"]
)

print(sim_df.head())

Total NaNs: 0
symbol           ABB  ADANIENSOL  ADANIENT  ADANIGREEN  ADANIPORTS  \
symbol                                                               
ABB         1.000000   -0.954815 -0.885369   -0.872418   -0.718033   
ADANIENSOL -0.954815    1.000000  0.983202    0.859659    0.890048   
ADANIENT   -0.885369    0.983202  1.000000    0.802971    0.955947   
ADANIGREEN -0.872418    0.859659  0.802971    1.000000    0.708686   
ADANIPORTS -0.718033    0.890048  0.955947    0.708686    1.000000   

symbol      ADANIPOWER  AMBUJACEM  APOLLOHOSP  ASIANPAINT      ATGL  ...  \
symbol                                                               ...   
ABB           0.177875   0.229141   -0.146894    0.994469 -0.595023  ...   
ADANIENSOL   -0.070649  -0.339133    0.115486   -0.962937  0.794157  ...   
ADANIENT     -0.018531  -0.378469    0.105488   -0.900126  0.879139  ...   
ADANIGREEN    0.306948  -0.634716   -0.325851   -0.914064  0.678157  ...   
ADANIPORTS    0.161462  -0.508437   -0.

In [7]:
peer_rows = []

for sym in sim_df.index:
    peers = (
        sim_df.loc[sym]
        .drop(sym)
        .sort_values(ascending=False)
        .head(5)
    )
    for rank, (peer_sym, score) in enumerate(peers.items(), start=1):
        peer_rows.append({
            "symbol": sym,
            "peer_symbol": peer_sym,
            "similarity_score": round(float(score), 4),
            "peer_rank": rank
        })

peer_df = pd.DataFrame(peer_rows)
peer_df.head()

,symbol,peer_symbol,similarity_score,peer_rank
0,ABB,ASIANPAINT,0.9945,1
1,ABB,BEL,0.9941,2
2,ABB,INFY,0.9925,3
3,ABB,BPCL,0.9732,4
4,ABB,IRCTC,0.9651,5


In [8]:
peer_df = peer_df.merge(
    dim_company[["symbol","company_name","sector"]],
    on="symbol", how="left"
).rename(columns={"company_name":"company_name", "sector":"sector"})

peer_df = peer_df.merge(
    dim_company[["symbol","company_name","sector"]],
    left_on="peer_symbol", right_on="symbol", how="left",
    suffixes=("", "_peer")
).rename(columns={
    "company_name_peer":"peer_company_name",
    "sector_peer":"peer_sector"
}).drop(columns=["symbol_peer"], errors="ignore")

In [9]:
peer_df[peer_df["symbol"]=="TCS"].sort_values("peer_rank")
peer_df[peer_df["symbol"]=="HDFCBANK"].sort_values("peer_rank")

,symbol,peer_symbol,similarity_score,peer_rank,company_name,sector,peer_company_name,peer_sector
190,HDFCBANK,BAJFINANCE,0.9546,1,HDFC Bank Ltd,Banking,Bajaj Finance Ltd,NBFC
191,HDFCBANK,HINDALCO,0.9300,2,HDFC Bank Ltd,Banking,Hindalco Industries Ltd,Metals
192,HDFCBANK,CIPLA,0.7359,3,HDFC Bank Ltd,Banking,Cipla Ltd,Pharma
193,HDFCBANK,COALINDIA,0.6808,4,HDFC Bank Ltd,Banking,Coal India Ltd,Energy
194,HDFCBANK,CHOLAFIN,0.6379,5,HDFC Bank Ltd,Banking,Cholamandalam Investment & Finance Company Ltd,NBFC


In [10]:
out_path = "../data/clean/peer_mapping.csv"
peer_df.to_csv(out_path, index=False)
out_path

'../data/clean/peer_mapping.csv'

In [11]:
peer_pivot = peer_df.pivot_table(
    index="symbol",
    columns="peer_rank",
    values="peer_company_name",
    aggfunc="first"
)
peer_pivot.head()

peer_rank,1,2,3,4,5
symbol,,,,,
ABB,Asian Paints Indian Multi-National Paint and C...,Bharat Electronics Ltd,Infosys Ltd,Bharat Petroleum Corporation Ltd,Indian Railway Catering & Tourism Corporation Ltd
ADANIENSOL,Macrotech Developers Ltd,Tata Power Company Ltd,Bajaj Finserv Ltd,Tata Steel Ltd,Info Edge (India) Ltd
ADANIENT,Tata Steel Ltd,Info Edge (India) Ltd,Jindal Steel & Power Ltd,Tata Power Company Ltd,Adani Energy Solutions Ltd
ADANIGREEN,Shriram Finance Ltd,IndusInd Bank Ltd,Axis Bank Ltd,Punjab National Bank,Bajaj Finserv Ltd
ADANIPORTS,ZOMATO,VBL,Avenue Supermarts Ltd,UNITDSPR,Jindal Steel & Power Ltd
